In [71]:
import os, sys
%pylab inline
import pandas as pd
import pickle
from importlib import reload

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


/camp/home/tootoos/working/tootoos/conda-envs/numpyro-env/lib/python3.10/site-packages/IPython/core/magics/pylab.py:166: UserWarning: pylab import has clobbered these variables: ['split']
`%matplotlib` prevents importing * from pylab and numpy
  warn("pylab import has clobbered these variables: %s"  % clobbered +


In [87]:
from glom_io_transform import paths
from glom_io_transform.data import odours; reload(odours)
from glom_io_transform.data import responses; reload(responses)

<module 'glom_io_transform.data.responses' from '/nemo/lab/schaefera/working/tootoos/git/glom-io-transform-release/glom_io_transform/data/responses.py'>

In [109]:
X0, Y0 = responses.load_experiments()

Loaded data from /nemo/lab/schaefera/working/tootoos/git/glom-io-transform-release/glom_io_transform/data/X0Y0_new.p


In [110]:
match_file = os.path.join(paths.data_root, "matched_roi_pairs_metadata.csv")
match_info = pd.read_csv(match_file)
match_info

,match_id,input_row,output_row,input_exp,output_exp,input_local_roi,output_local_roi,distance,input_x,input_y,output_x,output_y,correlation
0,1,7,110,C592,C525,7,0,0.000000,263.601700,124.995537,327.460368,262.235162,0.696483
1,2,2,113,C592,C525,2,3,11.258304,64.897366,92.753921,127.343159,218.824249,0.673008
2,3,51,111,C598,C525,15,1,8.498910,225.644123,200.074993,240.306466,222.831435,0.588918
3,4,8,114,C592,C525,8,4,19.196192,108.101185,221.121586,182.000000,342.000000,0.587893
4,5,65,148,C598,C531,29,18,15.476136,477.088086,281.980312,487.063158,309.243860,0.556860
5,6,59,0,C598,C444,23,0,0.000000,304.342045,238.625568,379.000000,269.334317,0.530443
6,7,45,97,C598,C519,9,3,22.114574,472.959224,103.879066,318.000000,54.734165,0.511400
7,8,0,39,C592,C470,0,9,22.599582,125.413019,105.557757,269.818095,254.000000,0.453562
8,9,35,159,C592,C537,35,9,16.342217,277.630053,254.831426,328.613600,299.944892,0.426633
9,10,3,33,C592,C470,3,3,7.708951,180.167150,189.804411,308.731565,330.000000,0.419286


In [118]:
def get_roi(X, which_exp, which_roi, match=None):
    """One ROI's responses, tagged with which one it is.

    experiment / roi_id / roi_label already ride along as scalar coordinates.
    What is missing is local_roi -- the POSITION within the experiment, which is
    what the matched-pair metadata indexes by, and which is not the same number
    as roi_id -- and the match the roi belongs to. Attaching them here means
    they survive a concat, so nothing downstream has to track list order.
    """
    for Xi in X:
        if which_exp in Xi.experiment.values:
            assert (Xi.experiment.values == which_exp).all(), \
                f"{which_exp} shares an array with other experiments; positional indexing is unsafe."
            assert 0 <= which_roi < Xi.sizes["roi"], \
                f"roi {which_roi} out of range for {which_exp}, which has {Xi.sizes['roi']} rois"
            Xret = Xi[which_roi].assign_coords(local_roi=which_roi)
            return Xret if match is None else Xret.assign_coords(match=match)
    raise ValueError(f"Experiment {which_exp} not found in X")

# Iterate each row
Xm, Ym = [], []
for index, row in match_info.iterrows():
    exp_input  = row["input_exp"]
    roi_input  = row["input_local_roi"]
    exp_output = row["output_exp"]
    roi_output = row["output_local_roi"]
    match      = row["match_id"] if "match_id" in row else index
    Xm.append(get_roi(X0, exp_input,  roi_input,  match=match))
    Ym.append(get_roi(Y0, exp_output, roi_output, match=match))
